# **<p align="center"> Ligand Analysis </p>**

## <ins> Description </ins>
### Explanation

<details open>
<summary> <font size="6"> <ins>Key Questions to Answer </ins> </font> </summary>

- Is there one ligand per subset?

- Is there one ligand per chain?

- Do the ligands have multiple conformations?

- Do ligands that are found in the same binding pocket have the same residue number? How are they different?

- What are the different labels that exist for ligands?


- image1: <img src="" width = 50%>

</details>

</br>



In [ ]:
# Load Key Libraries
from pathlib import Path
rootdir = Path("../../../..").resolve()
import sys
sys.path.insert(0, str(rootdir) )
from typing import Callable, Generator
# General Purpose Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Molecular Libraries
import gemmi 
from rdkit import Chem
import parasail
# Local Libraries
from xaidar.data.molecModels import loadPDB
from xaidar.data.molecModels import get_pdb_stats,  get_res_CoM , get_atom_coord
from xaidar.data.molecModels import flatten_pdb, createPDB, clear_empty
from xaidar.data.molecModels import sele_pdb, sele_Lig, sele_AA, sele_chain_name

/home/eoo22534/mydir/xaidar/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Test python objs
test_pdb_path = rootdir.joinpath("data/ev2a/fragalysis/02-chemofint/reference/A0926a.pdb")
test_pdb = loadPDB(test_pdb_path)
test_prot = sele_pdb( test_pdb, sele_AA)
test_prot_mon = sele_pdb( test_prot, sele_chain_name, chain_name = "A")
test_lig = sele_pdb( test_pdb, sele_Lig)
test_chain = flatten_pdb( test_prot_mon, "chain" )
test_res = test_chain[0][0] 
test_atom =  test_res[0]

---

## Q. What are the different labels that exist for ligands?


In [16]:
# Get the main data directory for crystallographic datasets
cryst_data_dir = rootdir.joinpath("data/ev2a/fragalysis/01-concatDir/crystallographic_files")
cryst_data_dir.exists()

# Get the main data directory for crystallographic datasets
align_data_dir = rootdir.joinpath("data/ev2a/fragalysis/01-concatDir/aligned_files")
align_data_dir.exists()

True

In [4]:
# Test flatten_structure with one dataset
dataset_path = cryst_data_dir.joinpath( "0501")
ref_prot = loadPDB( dataset_path.joinpath( f"A{dataset_path.name}a.pdb"))
get_pdb_stats( ref_prot)


####################
Number of models: 1
Number of chains in 1st Model: 4

Chain ID: A
	Number of Residues: 143
	Unique List of Non-A.A.: {'ZN', 'LIG'}
	Contains A.A.
Chain ID: B
	Number of Residues: 141
	Unique List of Non-A.A.: {'ZN'}
	Contains A.A.
Chain ID: C
	Number of Residues: 3
	Unique List of Non-A.A.: {'DMS'}
Chain ID: H
	Number of Residues: 416
	Unique List of Non-A.A.: {'HOH'}


In [ ]:
# Select organic molecules only
def sele_others( pdb: gemmi.Structure ):
    """Ensure that select from highest to lowest level of hierarchy, for 
    optimal results. I.e. first residues, then atoms."""
    def sele_others_pt1( lst_res: list[gemmi.Residue], level = False  ):
        if level: return "residue"
        if isinstance( lst_res, gemmi.Structure):
            lst_res = flatten_pdb( lst_res, "residue")
        return [ res for res in lst_res if 
                not gemmi.find_tabulated_residue(res.name).is_amino_acid() 
                and res.name not in ["HOH" ] 
                 ]

    def sele_others_pt2( lst_atom: list[gemmi.Atom], level = False ):
        if level: return "atom"
        if isinstance( lst_atom, gemmi.Structure):
            lst_res = flatten_pdb( lst_res, "atom")
        return [ atom for atom in lst_atom if 
                not atom.element.is_metal  ]

    others_res = sele_pdb( pdb,  sele_others_pt1, level= "residue") 
    others = sele_pdb( others_res, sele_others_pt2, level= "atom")
    return others


In [18]:
org_names = set()

for dataset in cryst_data_dir.iterdir():
    for subset in dataset.glob("*.pdb"):
        pdb = sele_others( loadPDB( subset) )
        org_names.update( [res.name for res in  flatten_pdb( sele_others( pdb ), "residue") ] )

for dataset in align_data_dir.iterdir():
    for dataset_path in dataset.glob(f"{dataset.name}.pdb"):
        pdb = sele_others( loadPDB( dataset_path) )
        org_names.update( [res.name for res in  flatten_pdb( sele_others( pdb ), "residue") ] )


print(org_names)


{'DMS', 'GOL', 'SO4', 'LIG'}


## Q. - Do ligands that are found in the same binding pocket have the same residue number? How are they different?

- ### Subpoint 1